# Anime Recommendation System

This notebook builds an end-to-end content-based recommendation system using the Anime Recommendations dataset. It includes data loading, EDA, preprocessing, content feature engineering, TF-IDF vectorization, cosine similarity, and recommendation generation.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import seaborn as sns

sns.set_style('whitegrid')

PROJECT_ROOT = Path().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.anime_recommender import (
    build_content_features,
    load_anime_data,
    plot_anime_correlation_heatmap,
    plot_anime_missing_values,
    plot_anime_rating_distribution,
    plot_anime_type_distribution,
    plot_episode_distribution,
    plot_most_rated_anime,
    plot_top_genres,
    plot_top_popular_anime,
    plot_top_rated_anime,
    plot_user_activity,
    plot_user_rating_distribution,
    preprocess_anime_data,
    recommend_anime,
)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Optional Kaggle Download

Run this only if `anime.csv` and `rating.csv` are not already present in `data/raw/`.

In [ ]:
# import subprocess
# subprocess.run(['kaggle', 'datasets', 'download', '-d', 'CooperUnion/anime-recommendations-database'], check=True)
# unzip the downloaded file and place anime.csv and rating.csv in data/raw/


## 3. Load Data

In [ ]:
anime_path = PROJECT_ROOT / 'data' / 'raw' / 'anime.csv'
rating_path = PROJECT_ROOT / 'data' / 'raw' / 'rating.csv'

if not anime_path.exists() or not rating_path.exists():
    raise FileNotFoundError('Place anime.csv and rating.csv inside data/raw/.')

anime_raw, ratings_raw = load_anime_data(str(anime_path), str(rating_path))
anime_raw.head()

In [ ]:
ratings_raw.head()

In [ ]:
anime, ratings, merged = preprocess_anime_data(anime_raw, ratings_raw)
print('Anime shape:', anime.shape)
print('Ratings shape after removing -1:', ratings.shape)
print('Merged shape:', merged.shape)

## 4. EDA and Visualizations

In [ ]:
anime.describe(include='all').T.head(15)

In [ ]:
plot_anime_missing_values(anime)
plot_anime_rating_distribution(anime)
plot_top_popular_anime(anime)
plot_anime_type_distribution(anime)
plot_episode_distribution(anime)
plot_user_rating_distribution(ratings)
plot_top_rated_anime(merged)
plot_most_rated_anime(merged)
plot_anime_correlation_heatmap(anime)
plot_top_genres(anime)
plot_user_activity(ratings)

## 5. Check Distributions and Skewness

In [ ]:
numeric_cols = anime.select_dtypes(include=['int64', 'float64']).columns.tolist()
anime[numeric_cols].skew().sort_values(ascending=False)

The recommender itself is based on text-like content features rather than scaled numeric-only features, so the most important preprocessing here is cleaning missing values and creating useful metadata-based content strings.

## 6. Create Content Features

In [ ]:
anime_content = build_content_features(anime)
anime_content[['name', 'genre', 'type', 'rating', 'members', 'content']].head()

## 7. TF-IDF and Cosine Similarity

In [ ]:
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(anime_content['content'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print('TF-IDF matrix shape:', tfidf_matrix.shape)

## 8. Recommendation Function

In [ ]:
recommend_anime('Naruto', anime_content, cosine_sim, top_n=10)

In [ ]:
recommend_anime('Death Note', anime_content, cosine_sim, top_n=10)

## 9. Conclusion

This project creates a simple and interpretable content-based recommendation engine using genre, type, rating bands, and popularity bands. Similar anime are identified through cosine similarity on TF-IDF-transformed content features.